# Catalog-Linked Database: Vended Credentials Lab Guide

This notebook walks you through setting up a **Catalog-Linked Database (CLD)** using AWS Glue Iceberg REST catalog with **vended credentials** (Lake Formation credential vending).

The setup uses two IAM roles:
- **Snowflake role** — SigV4 auth for the Glue REST API
- **Lake Formation role** — S3 credential vending via Lake Formation

**Prerequisites:**
- AWS account with Glue, Lake Formation, and IAM access
- An existing Glue database with Iceberg tables
- An S3 bucket containing the Iceberg data files
- Snowflake ACCOUNTADMIN (or a role with CREATE INTEGRATION + CREATE DATABASE privileges)

> ### IMPORTANT
> Setup Glue DB with Lake Formation on your AWS account using `task bronze:all` from within [Lab Sources](https://github.com/Snowflake-Labs/sfguide-lakehouse-iceberg-production-pipelines) repo

---

## Step 1: Configure Your Environment

Set the variables below to match your environment. All subsequent SQL cells reference these variables via Jinja templating — change them once and everything updates.

> ### 🛑 STOP — Update the variables above before proceeding!STOP — Update the variables below before proceeding!
> 
> Make sure you have set **all variables** to match your environment, then **run the cell below** before continuing. All subsequent SQL cells depend on these values.

In [ ]:
SF_ROLE = 'ACCOUNTADMIN'
CATALOG_INTEGRATION_NAME = 'ksampath_glue_rest_catalog_int'
GLUE_DB = 'ksampath_balloon_pops'
GLUE_NAMESPACE = GLUE_DB
AWS_ACCOUNT_ID = '849350360261'
SNOWFLAKE_IAM_ROLE = 'ksampath_snowflake_glue_catalog_read'
AWS_REGION = 'us-west-2'
LF_IAM_ROLE = 'ksampath-lf-data-access'
S3_BUCKET = 'ksampath-balloon-bronze'
CLD_DATABASE = 'balloon_game_events'
GLUE_TABLE = 'balloon_game_events'

---

## AWS Prerequisites

Vended credentials require a **two-role setup** in AWS:

1. **Snowflake IAM role** — Snowflake assumes this role via SigV4 to call the Glue REST API and request temporary credentials from Lake Formation.
2. **Lake Formation IAM role** — Lake Formation assumes this role to vend short-lived S3 credentials scoped to the registered data location.

The cell below renders the IAM policies and AWS CLI commands using your variables. Run the Variables cell first, then run the cell below to get copy-paste-ready commands for your AWS environment.

In [ ]:
import json
from IPython.display import display, Markdown

snowflake_role_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "GlueCatalogAccess",
            "Effect": "Allow",
            "Action": [
                "glue:GetCatalog", "glue:GetCatalogs",
                "glue:GetDatabase", "glue:GetDatabases",
                "glue:GetTable", "glue:GetTables"
            ],
            "Resource": [
                f"arn:aws:glue:{AWS_REGION}:{AWS_ACCOUNT_ID}:catalog",
                f"arn:aws:glue:{AWS_REGION}:{AWS_ACCOUNT_ID}:catalog/*",
                f"arn:aws:glue:{AWS_REGION}:{AWS_ACCOUNT_ID}:database/{GLUE_DB}",
                f"arn:aws:glue:{AWS_REGION}:{AWS_ACCOUNT_ID}:table/{GLUE_DB}/*"
            ]
        },
        {
            "Sid": "LakeFormationCredentialVending",
            "Effect": "Allow",
            "Action": [
                "lakeformation:GetDataAccess",
                "lakeformation:GetTemporaryGlueTableCredentials",
                "lakeformation:GetTemporaryGluePartitionCredentials"
            ],
            "Resource": "*"
        }
    ]
}, indent=4)

lf_role_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "S3ReadForIceberg",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:GetObjectVersion", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{S3_BUCKET}",
                f"arn:aws:s3:::{S3_BUCKET}/*"
            ]
        }
    ]
}, indent=4)

lf_trust_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "LakeFormationAssumeRole",
            "Effect": "Allow",
            "Principal": {"Service": "lakeformation.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}, indent=4)

display(Markdown(f"""
---

## Prerequisites (AWS side — do these BEFORE running the notebook)

### 1. Create Snowflake IAM role: `{SNOWFLAKE_IAM_ROLE}`

```bash
# Create the role (empty trust policy — updated after DESC INTEGRATION)
aws iam create-role \\
  --role-name {SNOWFLAKE_IAM_ROLE} \\
  --assume-role-policy-document '{json.dumps({"Version": "2012-10-17", "Statement": []})}'

# Attach permissions policy
aws iam put-role-policy \\
  --role-name {SNOWFLAKE_IAM_ROLE} \\
  --policy-name GlueLakeFormationAccess \\
  --policy-document '{snowflake_role_policy}'
```

<details><summary>Permissions policy JSON</summary>

```json
{snowflake_role_policy}
```
</details>

### 2. Create Lake Formation IAM role: `{LF_IAM_ROLE}`

```bash
# Create the role with LF trust policy
aws iam create-role \\
  --role-name {LF_IAM_ROLE} \\
  --assume-role-policy-document '{lf_trust_policy}'

# Attach S3 read policy
aws iam put-role-policy \\
  --role-name {LF_IAM_ROLE} \\
  --policy-name S3ReadAccess \\
  --policy-document '{lf_role_policy}'
```

<details><summary>Permissions policy JSON</summary>

```json
{lf_role_policy}
```
</details>

<details><summary>Trust policy JSON</summary>

```json
{lf_trust_policy}
```
</details>

### 3. Lake Formation setup

```bash
# Register S3 location with LF role
aws lakeformation register-resource \\
  --resource-arn "arn:aws:s3:::{S3_BUCKET}" \\
  --role-arn "arn:aws:iam::{AWS_ACCOUNT_ID}:role/{LF_IAM_ROLE}" \\
  --hybrid-access-enabled \\
  --region {AWS_REGION}

# Disable "Use only IAM access control" on the database
aws glue update-database \\
  --name {GLUE_DB} \\
  --database-input '{json.dumps({"Name": GLUE_DB, "CreateTableDefaultPermissions": []})}' \\
  --region {AWS_REGION}

# Grant Snowflake role access via Lake Formation
aws lakeformation grant-permissions \\
  --principal DataLakePrincipalIdentifier="arn:aws:iam::{AWS_ACCOUNT_ID}:role/{SNOWFLAKE_IAM_ROLE}" \\
  --resource '{json.dumps({"Database": {"Name": GLUE_DB}})}' \\
  --permissions "DESCRIBE" \\
  --region {AWS_REGION}

aws lakeformation grant-permissions \\
  --principal DataLakePrincipalIdentifier="arn:aws:iam::{AWS_ACCOUNT_ID}:role/{SNOWFLAKE_IAM_ROLE}" \\
  --resource '{json.dumps({"Table": {"DatabaseName": GLUE_DB, "TableWildcard": {}}})}' \\
  --permissions "SELECT" "DESCRIBE" \\
  --region {AWS_REGION}
```
"""))

---

## Step 2: Set Role and Create Catalog Integration

In [ ]:
%%sql -r use_role_result
USE ROLE {{SF_ROLE}};

Create a catalog integration that connects Snowflake to AWS Glue via the Iceberg REST API. This uses SigV4 authentication with your Snowflake IAM role and enables `VENDED_CREDENTIALS` so Lake Formation handles S3 data access.

In [ ]:
%%sql -r create_catalog_int_result
CREATE OR REPLACE CATALOG INTEGRATION {{CATALOG_INTEGRATION_NAME}}
  CATALOG_SOURCE = ICEBERG_REST
  TABLE_FORMAT = ICEBERG
  CATALOG_NAMESPACE = '{{GLUE_NAMESPACE}}'
  REST_CONFIG = (
    CATALOG_URI = 'https://glue.{{AWS_REGION}}.amazonaws.com/iceberg'
    CATALOG_API_TYPE = AWS_GLUE
    CATALOG_NAME = '{{AWS_ACCOUNT_ID}}'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  REST_AUTHENTICATION = (
    TYPE = SIGV4
    SIGV4_IAM_ROLE = 'arn:aws:iam::{{AWS_ACCOUNT_ID}}:role/{{SNOWFLAKE_IAM_ROLE}}'
    SIGV4_SIGNING_REGION = '{{AWS_REGION}}'
  )
  ENABLED = TRUE;

---

## Step 2b: Get Integration Details for Trust Policy

In [ ]:
%%sql -r desc_integration_result
DESC INTEGRATION {{CATALOG_INTEGRATION_NAME}};

In [ ]:
import json
from IPython.display import display, Markdown

props = dict(zip(desc_integration_result['property'], desc_integration_result['property_value']))
iam_user_arn = props.get('API_AWS_IAM_USER_ARN', '<run DESC INTEGRATION first>')
external_id = props.get('API_AWS_EXTERNAL_ID', '<run DESC INTEGRATION first>')

trust_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "SnowflakeCatalogIntegration",
            "Effect": "Allow",
            "Principal": {"AWS": iam_user_arn},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {
                    "sts:ExternalId": external_id
                }
            }
        }
    ]
}, indent=4)

display(Markdown(f"""
## STOP — Update Snowflake IAM Role Trust Policy

| Property | Value |
|---|---|
| `API_AWS_IAM_USER_ARN` | `{iam_user_arn}` |
| `API_AWS_EXTERNAL_ID` | `{external_id}` |

```bash
aws iam update-assume-role-policy \\
  --role-name {SNOWFLAKE_IAM_ROLE} \\
  --policy-document '{trust_policy}'
```

<details><summary>Trust policy JSON</summary>

```json
{trust_policy}
```
</details>

**Wait 1-2 minutes** for IAM propagation, then continue.

**WARNING:** Do NOT re-run the CREATE CATALOG INTEGRATION cell after this — it rotates the external ID.
"""))

---

## Step 3: Grant Integration Usage and Create CLD

### 3.1 Grant Integration Usage

In [ ]:
%%sql -r grant_usage_result
GRANT USAGE ON INTEGRATION {{CATALOG_INTEGRATION_NAME}} TO ROLE {{SF_ROLE}};

### 3.2 Lake Formation Settings for Vended Credentials



Before creating the Catalog-Linked Database, ensure the following Lake Formation settings are configured correctly. Without these, the CLD will fail with:

```
SQL Execution Error: Failed to retrieve credentials from the Catalog for table ...
Please ensure that the catalog vends credentials and retry.
```

**Data Lake Location (Lake Formation > Data lake locations):**

| Setting | Required Value | Notes |
|---|---|---|
| S3 location | `s3://<your-bucket>` | Must cover the path where Iceberg data files reside |
| IAM role | Lake Formation role (e.g. `ksampath-lf-data-access`) | Must be assumable by `lakeformation.amazonaws.com` with S3 read access |
| Permission mode | **Lake Formation** | **Not Hybrid** — this is the most common cause of vended credentials failures |
| Data Catalog Federation | Auto-managed | Selecting "Lake Formation" mode automatically disables this checkbox in the UI; it may be used internally |

**Glue Database settings:**
- "Use only IAM access control for new tables" — does **not** affect vended credentials (can be checked or unchecked)

**Lake Formation Data Permissions (SELECT/DESCRIBE on tables):**
- These only control which schemas/tables appear in the CLD during auto-discovery
- They are **not** required for vended credentials to function

**Critical: Applying Lake Formation changes to an existing CLD**
- `ALTER DATABASE ... RESUME DISCOVERY` does **NOT** re-establish the catalog connection — it only retries table/schema discovery
- After changing any Lake Formation setting, you **must** run `CREATE OR REPLACE DATABASE ... LINKED_CATALOG` to re-establish the link
- After recreating the database, re-run `GRANT USAGE ON INTEGRATION` to the owner role

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

images = [
    ('lab/images/aws_lf_data_lake_settings.png', 'Data Lake Location Settings'),
    ('lab/images/aws_lf_database_settings.png', 'Database Settings'),
    ('lab/images/aws_lf_data_permissions.png', 'Data Permissions'),
]

for ax, (path, title) in zip(axes, images):
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

### 3.3 Create Catalog-Linked Database

In [ ]:
%%sql -r create_cld_result
CREATE OR REPLACE DATABASE {{CLD_DATABASE}}
  COMMENT = 'CLD: Glue Iceberg with vended credentials (two-role setup)'
  LINKED_CATALOG = (
    CATALOG = '{{CATALOG_INTEGRATION_NAME}}'
  );

---

## Step 4: Check Link Status and Resume Discovery

Check the CLD Link status to verify link us up and syncing

In [ ]:
%%sql -r link_status_result
SELECT SYSTEM$CATALOG_LINK_STATUS('{{CLD_DATABASE}}');

If the link status shows failures, use `RESUME DISCOVERY` to retry table/schema discovery. Note: this only retries discovery — if AWS-side settings (Lake Formation, IAM) changed, you must `CREATE OR REPLACE DATABASE` instead.

In [ ]:
%%sql -r resume_discovery_result
ALTER DATABASE {{CLD_DATABASE}} RESUME DISCOVERY;

---

## Step 5: Validate

List the schemas discovered in the linked Glue namespace. These are auto-synced from the Glue catalog.

In [ ]:
%%sql -r show_schemas_result
SHOW SCHEMAS IN DATABASE {{CLD_DATABASE}};

List the Iceberg tables discovered in the linked Glue namespace. These are auto-synced from the Glue catalog.

In [ ]:
%%sql -r show_tables_result
SHOW ICEBERG TABLES IN SCHEMA {{CLD_DATABASE}}."{{GLUE_NAMESPACE}}";

Query the Iceberg table data directly through Snowflake. The `event` column contains JSON — we use `PARSE_JSON` to extract individual fields.

In [ ]:
%%sql -r sample_data_result
SELECT
  PARSE_JSON(event):player::STRING AS player,
  PARSE_JSON(event):balloon_color::STRING AS balloon_color,
  PARSE_JSON(event):score::INTEGER AS score,
  PARSE_JSON(event):event_ts::TIMESTAMP_TZ AS event_ts
FROM {{CLD_DATABASE}}."{{GLUE_NAMESPACE}}"."{{GLUE_TABLE}}"
LIMIT 10;

---

## Cleanup

### Snowflake cleanup

In [ ]:
%%sql -r cleanup_result
USE ROLE {{SF_ROLE}};
DROP DATABASE IF EXISTS {{CLD_DATABASE}};
DROP CATALOG INTEGRATION IF EXISTS {{CATALOG_INTEGRATION_NAME}};

In [ ]:
import json
from IPython.display import display, Markdown

display(Markdown(f"""
### AWS cleanup

```bash
# Revoke Lake Formation permissions
aws lakeformation revoke-permissions \\
  --principal DataLakePrincipalIdentifier="arn:aws:iam::{AWS_ACCOUNT_ID}:role/{SNOWFLAKE_IAM_ROLE}" \\
  --resource '{json.dumps({"Database": {"Name": GLUE_DB}})}' \\
  --permissions "DESCRIBE" \\
  --region {AWS_REGION}

aws lakeformation revoke-permissions \\
  --principal DataLakePrincipalIdentifier="arn:aws:iam::{AWS_ACCOUNT_ID}:role/{SNOWFLAKE_IAM_ROLE}" \\
  --resource '{json.dumps({"Table": {"DatabaseName": GLUE_DB, "TableWildcard": {}}})}' \\
  --permissions "SELECT" "DESCRIBE" \\
  --region {AWS_REGION}

# Deregister S3 location
aws lakeformation deregister-resource \\
  --resource-arn "arn:aws:s3:::{S3_BUCKET}" \\
  --region {AWS_REGION}

# Delete IAM roles (optional)
aws iam delete-role-policy --role-name {SNOWFLAKE_IAM_ROLE} --policy-name GlueLakeFormationAccess
aws iam delete-role --role-name {SNOWFLAKE_IAM_ROLE}

aws iam delete-role-policy --role-name {LF_IAM_ROLE} --policy-name S3ReadAccess
aws iam delete-role --role-name {LF_IAM_ROLE}
```
"""))

---

## Troubleshooting



### "Failed to retrieve credentials from the Catalog"

If the link status shows this error:
```json
{"failureDetails":[{"errorCode":"094120","errorMessage":"SQL Execution Error: Failed to retrieve credentials from the Catalog for table ..."}],"executionState":"RUNNING"}
```

**Root cause:** The Lake Formation Data Lake Location is not configured correctly for credential vending.

**Required Lake Formation setting (Data Lake Locations):**
- Permission mode must be **Lake Formation** (not Hybrid)
- The location must be registered with an IAM role that Lake Formation can assume and that has S3 read access

**Important notes:**
- `ALTER DATABASE ... RESUME DISCOVERY` does **NOT** re-establish the catalog connection. It only retries table/schema discovery.
- After changing Lake Formation settings, you **must** run `CREATE OR REPLACE DATABASE ... LINKED_CATALOG` to re-establish the link.
- After `CREATE OR REPLACE DATABASE`, you must re-run `GRANT USAGE ON INTEGRATION` (see steps above).